# 04 · EDA del ajuste fino — Etapa 2 reformulada sobre M3DI-*test* ($n = 10\,000$)

**Pregunta.** ¿Cuánto del error zero-shot de la etapa 1 era **formato** y cuánto **percepción**? ¿Qué modelo aprende más (con train completo, 125 000 imágenes), y si lo aprendido **compone** (forma × posición) o se memoriza?

Lee `runs/` en solo lectura y escribe en `reports/{figures,tables}/eda_finetuning/`. El diseño está en `docs/ETAPA2B_REFORMULACION.md`. Cada sección se salta con un aviso si sus runs todavía no existen, así que el notebook se puede correr en cualquier momento del plan.

| § | Contenido | Runs que usa |
|---|---|---|
| 1 | Configuración, inventario e integridad | todos |
| 2 | Curvas de entrenamiento (loss, lr, img/s, memoria) | `train_*` |
| 3 | Curvas de validación y paso elegido | `val_log.jsonl` |
| 4 | Test: zero-shot p1 frente a ajustado (IC de Wilson, McNemar) | `stage2b_main` + `stage2b_test` |
| 5 | Qué cambió: confusiones de forma, posición por celda, transiciones | ídem |
| 6 | Split composicional: combinaciones vistas frente a retiradas | `stage2b_compositional` |
| 7 | Sondas lineales: lo que el modelo *sabe* frente a lo que *dice* | `stage2b_probe`, `stage2b_probe_ft` |
| 8 | Controles: objetivo original, bf16, semilla | `stage2b_ablations` |
| 9 | Costo y formato de las respuestas | todos |
| 10 | Ejemplos, síntesis y exportación | — |

**Notación.** Para la imagen $i$ y el atributo $a$, $k_{i,a}=\mathbb 1[\hat z_{i,a}=z_{i,a}]$ (omitir es fallar), con el extractor de `vlmfid.eval.attributes`, el mismo de la etapa 1. Métricas: forma, posición $(x,y)$, color (categoría exacta), color ±1, nombre exacto, `shape_pos` $=\tfrac12(k_\text{forma}+k_\text{pos})$ (selección del checkpoint), macro $=\tfrac13(k_\text{forma}+k_\text{pos}+k_{\text{color}\pm1})$ y `macro_cat` (con color exacto). Con el objetivo canónico el **nombre exacto es 0 por construcción**.

In [ ]:
import json
import math
import re
import sys
import textwrap
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

try:
    from vlmfid.paths import ROOT, RUNS
except ImportError:                      # kernel sin el paquete instalado en modo editable
    sys.path.insert(0, str(Path.cwd().resolve().parent / 'src'))
    from vlmfid.paths import ROOT, RUNS
from vlmfid.data import data_root, load_manifest
from vlmfid.data.targets import HELD_OUT_COMBOS, canonical_caption, combo, is_held_out
from vlmfid.eval.attributes import COLOR_CATS, SHAPE_NAMES, XPOS_NAMES, YPOS_NAMES, extract, score, truth_from_record

# ── Parámetros ───────────────────────────────────────────────────────────────
SPLIT, VARIANT = 'test', 'base'
LIMIT = 10_000            # mismo conjunto que el zero-shot de la etapa 1 (data.limit=10000 o null)
SUBSET_SEED = 0
PROMPT = 'p1_attributes'  # prompt común de la etapa 2 (y su línea base zero-shot)
ALPHA = 0.05
SEED = 20261023
N_EXAMPLES = 3            # ejemplos cualitativos por modelo (§10)
TAG = 'eda_finetuning'
FIG_DIR = ROOT / 'reports' / 'figures' / TAG
TAB_DIR = ROOT / 'reports' / 'tables' / TAG
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

MODEL_LABELS = {                         # model.name -> etiqueta; el orden define el de figuras y tablas
    'vljepa_open_msrvtt': 'VL-JEPA',
    'qwen35_08b_base': 'Qwen3.5 Base',
    'internvl3_1b': 'InternVL3 1B',
    'llava_ov_05b': 'LLaVA-OV 0.5B',
}
MODEL_ORDER = list(MODEL_LABELS)
KIND_ORDER = ['zero-shot', 'principal', 'composicional', 'original', 'bf16', 'semilla', 'zero-shot bf16']
METRICS = ['shape', 'x', 'y', 'pos', 'color', 'color_adj', 'name', 'shape_pos', 'macro', 'macro_cat']
BINARY = ['shape', 'x', 'y', 'pos', 'color', 'color_adj', 'name']
MLAB = {'shape': 'Forma', 'x': 'x', 'y': 'y', 'pos': 'Posición (x,y)', 'color': 'Color', 'color_adj': 'Color ±1',
        'name': 'Nombre exacto', 'shape_pos': 'Forma+pos', 'macro': 'Índice macro', 'macro_cat': 'Macro (color exacto)'}
MAIN_METRICS = ['shape', 'pos', 'color', 'color_adj', 'shape_pos', 'macro']

RNG = np.random.default_rng(SEED)
Z = 1.959963984540054
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 120)
warnings.filterwarnings('ignore', category=FutureWarning)
import logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
print(f'ROOT = {ROOT}\nRUNS = {RUNS}\nsalidas -> {FIG_DIR.relative_to(ROOT)}, {TAB_DIR.relative_to(ROOT)}')

In [ ]:
# ── Estilo (mismo sistema que los notebooks 02-03) y utilidades estadísticas ──
SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#4a3aa7', '#e34948']
INK, INK2, INK3, GRID, SURF, NEUTRAL = '#0b0b0b', '#52514e', '#8a8984', '#e6e5e1', '#fcfcfb', '#c9c8c2'
SEQ = mcolors.LinearSegmentedColormap.from_list(
    'seq_blue', ['#f4f8fe', '#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#256abf', '#184f95', '#0d366b'])
DIV = mcolors.LinearSegmentedColormap.from_list('div_red_blue', ['#c23a39', '#ee9190', '#f0efec', '#6da7ec', '#184f95'])
MARKERS = ['o', 's', '^', 'D', 'v', 'P', 'X', '*']
MODEL_COLORS = {m: SERIES[i % len(SERIES)] for i, m in enumerate(MODEL_ORDER)}
MODEL_MARKERS = {m: MARKERS[i % len(MARKERS)] for i, m in enumerate(MODEL_ORDER)}

plt.rcParams.update({
    'figure.facecolor': SURF, 'axes.facecolor': SURF, 'savefig.facecolor': SURF,
    'axes.edgecolor': INK3, 'axes.labelcolor': INK2, 'axes.titlecolor': INK,
    'axes.titlesize': 10.5, 'axes.titleweight': 'semibold', 'axes.labelsize': 9.5,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'xtick.color': INK2, 'ytick.color': INK2, 'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5,
    'legend.frameon': False, 'legend.fontsize': 8.5, 'font.size': 9.5,
    'lines.linewidth': 2, 'lines.markersize': 7, 'figure.dpi': 110,
})
PCT = plt.FuncFormatter(lambda v, _: f'{v:.0%}')


def savefig(fig, name):
    for ext in ('png', 'pdf'):
        fig.savefig(FIG_DIR / f'{name}.{ext}', dpi=200, bbox_inches='tight')


def save_table(df, name, index=True, float_format='%.3f'):
    df.to_csv(TAB_DIR / f'{name}.csv', index=index)
    try:
        df.to_latex(TAB_DIR / f'{name}.tex', index=index, float_format=float_format, escape=True, na_rep='--')
    except Exception as e:                       # to_latex requiere jinja2
        print(f'[aviso] {name}.tex no generado: {e}')


def heat(ax, M, xlabels, ylabels, fmt='{:.0%}', vmin=0.0, vmax=1.0, cmap=SEQ, fontsize=8):
    M = np.asarray(M, dtype=float)
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    ax.imshow(np.where(np.isfinite(M), M, np.nan), cmap=cmap, norm=norm, aspect='auto')
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M[i, j]
            if np.isfinite(v):
                r, g, b, _ = cmap(norm(v))
                lum = 0.2126 * r + 0.7152 * g + 0.0722 * b
                ax.text(j, i, fmt.format(v), ha='center', va='center', fontsize=fontsize, color='white' if lum < 0.5 else INK)
            else:
                ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, color=GRID, lw=0))
                ax.text(j, i, '–', ha='center', va='center', fontsize=fontsize, color=INK3)
    ax.set_xticks(range(M.shape[1]), xlabels)
    ax.set_yticks(range(M.shape[0]), ylabels)
    ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.tick_params(length=0)


def wilson(k, n, z=Z):
    k, n = np.asarray(k, float), np.asarray(n, float)
    with np.errstate(invalid='ignore', divide='ignore'):
        p = k / n
        den = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / den
        hw = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / den
    return c - hw, c + hw


def mcnemar_exact(b, c):
    # p-valor bilateral exacto de McNemar: bajo H0, b ~ Bin(b + c, 1/2).
    n = int(b + c)
    if n == 0:
        return 1.0
    k = int(min(b, c))
    logs = [math.lgamma(n + 1) - math.lgamma(i + 1) - math.lgamma(n - i + 1) - n * math.log(2) for i in range(k + 1)]
    m = max(logs)
    return min(1.0, 2 * math.exp(m + math.log(sum(math.exp(l - m) for l in logs))))


def holm(p):
    p = np.asarray(p, float)
    order, m = np.argsort(p), len(p)
    adj, run = np.empty(m), 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i])
        adj[i] = min(1.0, run)
    return adj


def read_jsonl(path):
    rows = []
    if not path.exists():
        return rows
    with path.open(encoding='utf-8') as f:
        for line in f:
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass                              # línea truncada por una caída
    return rows


def read_json(path):
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else {}


def read_yaml(path):
    return (yaml.safe_load(path.read_text(encoding='utf-8')) or {}) if path.exists() else {}


def skip(msg):
    print(f'[omitido] {msg}')

## 1. Inventario e integridad

Runs de **entrenamiento** (`runs/train_*`), clasificados por su configuración y no por su nombre:

| clase | criterio |
|---|---|
| principal | objetivo canónico, train completo, NF4, semilla 0 |
| composicional | `data.holdout = compositional` |
| original / bf16 / semilla | control de `stage2b_ablations` |
| humo | `data.val_limit ≠ 500` (se excluye) |

Runs de **test**: `stage = finetuned` con `model.adapter_path` (el adaptador identifica su run de entrenamiento) y las líneas base zero-shot de la etapa 1 con el mismo prompt (p1) y las mismas 10 000 imágenes.

In [ ]:
def classify(cfg):
    d, t = cfg.get('data') or {}, cfg.get('train') or {}
    if d.get('val_limit') != 500 or t.get('max_steps'):
        return 'humo'
    fmt = (cfg.get('target') or {}).get('format', 'original')
    if d.get('holdout'):
        return 'composicional'
    if fmt == 'original':
        return 'original'
    if (cfg.get('model') or {}).get('quant') != 'nf4':
        return 'bf16'
    if int(cfg.get('seed') or 0) != 0:
        return 'semilla'
    return 'principal'


train_rows = []
for cfg_path in sorted(RUNS.glob('train_*/config.yaml')):
    d = cfg_path.parent
    cfg = read_yaml(cfg_path)
    model = (cfg.get('model') or {}).get('name')
    if model not in MODEL_LABELS:
        continue
    summ, st, best = read_json(d / 'summary.json'), read_json(d / 'status.json'), read_json(d / 'best.json')
    mi = st.get('model_info') or {}
    data, tr = cfg.get('data') or {}, cfg.get('train') or {}
    train_rows.append({
        'exp_id': cfg.get('exp_id', d.name), 'model': model, 'kind': classify(cfg),
        'target': (cfg.get('target') or {}).get('format', 'original'), 'holdout': data.get('holdout'),
        'n_train': summ.get('n_train') or mi.get('n_train'),
        'epochs': tr.get('epochs'), 'quant': (cfg.get('model') or {}).get('quant'), 'seed': cfg.get('seed'),
        'prompt': (cfg.get('prompt') or {}).get('id'), 'selection': tr.get('selection_metric'),
        'estado': st.get('status', 'desconocido'), 'paso': st.get('step'), 'pasos_total': st.get('total_steps'),
        'mejor_paso': best.get('step'), 'mejor_valor': best.get('value'),
        'train_h': summ.get('train_hours'), 'eval_h': summ.get('eval_hours'),
        'lora': summ.get('trainable_lora') or mi.get('trainable_lora'),
        'conector': summ.get('trainable_connector') or mi.get('trainable_connector'),
        'adapter_best': (d / 'adapter_best').exists(), 'dir': d,
    })
train_runs = pd.DataFrame(train_rows)
HAVE_TRAIN = len(train_runs) > 0
if HAVE_TRAIN:
    train_runs = train_runs[train_runs['kind'] != 'humo'].copy()
    train_runs['label'] = train_runs['model'].map(MODEL_LABELS)
    train_runs['_m'] = train_runs['model'].map(MODEL_ORDER.index)
    train_runs['_k'] = train_runs['kind'].map(lambda k: KIND_ORDER.index(k) if k in KIND_ORDER else 99)
    train_runs = train_runs.sort_values(['_k', '_m', 'n_train']).drop(columns=['_m', '_k']).reset_index(drop=True)
    display(train_runs.drop(columns=['dir']))
else:
    skip(f'no hay runs de entrenamiento en {RUNS} (correr stage2b_main.txt)')
TRAIN_KIND = dict(zip(train_runs['exp_id'], train_runs['kind'])) if HAVE_TRAIN else {}
TRAIN_N = dict(zip(train_runs['exp_id'], train_runs['n_train'])) if HAVE_TRAIN else {}
MAIN_TRAIN = {r.model: r for r in train_runs.itertuples() if r.kind == 'principal'} if HAVE_TRAIN else {}

In [ ]:
# ── Verdad de test (las mismas 10 000 imágenes del zero-shot) ─────────────────
records = load_manifest(SPLIT, limit=LIMIT, subset_seed=SUBSET_SEED, variant=VARIANT)
TRUTH = {r['id']: truth_from_record(r) for r in records}
truth = pd.DataFrame([{
    'id': r['id'], 'image_path': r['image_path'], 'reference': r['caption'], 'canonical': canonical_caption(r),
    'held_out': is_held_out(r), 'shape_code': combo(r)[0], 'pos_code': combo(r)[1], **TRUTH[r['id']],
} for r in records]).set_index('id')
EXPECTED = set(truth.index)
print(f'{SPLIT}: {len(truth):,} imágenes · combinaciones retiradas en el split composicional: {len(HELD_OUT_COMBOS)}/63 '
      f'-> {truth["held_out"].mean():.1%} de las imágenes de test')

# ── Predicciones de test: ajustadas (stage=finetuned) y zero-shot p1 ───────────
meta_rows, frames = [], []
for cfg_path in sorted(RUNS.glob('*/config.yaml')):
    cfg = read_yaml(cfg_path)
    d = cfg_path.parent
    data, mcfg = cfg.get('data') or {}, cfg.get('model') or {}
    if data.get('split') != SPLIT or cfg.get('stage') not in ('zeroshot', 'finetuned'):
        continue
    if (cfg.get('prompt') or {}).get('id') != PROMPT or mcfg.get('name') not in MODEL_LABELS:
        continue
    if data.get('limit') not in (None, LIMIT) or data.get('variant', 'base') != VARIANT:
        continue
    if cfg['stage'] == 'finetuned':
        if not mcfg.get('adapter_path'):
            continue
        train_id = Path(str(mcfg['adapter_path'])).parent.name
        kind = TRAIN_KIND.get(train_id, 'desconocido')
    else:
        train_id, kind = None, ('zero-shot' if mcfg.get('quant') == 'nf4' else 'zero-shot bf16')
    st = read_json(d / 'status.json')
    preds = read_jsonl(d / 'predictions.jsonl')
    ids = [p['id'] for p in preds]
    meta_rows.append({'exp_id': cfg.get('exp_id', d.name), 'model': mcfg['name'], 'kind': kind, 'train_id': train_id,
                      'n_train': TRAIN_N.get(train_id), 'estado': st.get('status', 'desconocido'), 'filas': len(ids),
                      'faltantes': len(EXPECTED - set(ids)), 'img_por_s': st.get('img_per_s')})
    frames.append(pd.DataFrame({'exp_id': meta_rows[-1]['exp_id'], 'id': ids,
                                'prediction': [p.get('prediction') or '' for p in preds]}))

test_meta = pd.DataFrame(meta_rows)
HAVE_TEST = len(test_meta) > 0
if HAVE_TEST:
    test_meta['cobertura'] = 1 - test_meta['faltantes'] / len(EXPECTED)
    # dos zero-shot del mismo modelo (limit=10000 y null): se conserva el de mayor cobertura
    test_meta = (test_meta.sort_values('cobertura', ascending=False)
                 .drop_duplicates(['model', 'kind', 'train_id'], keep='first'))
    test_meta['label'] = test_meta['model'].map(MODEL_LABELS)
    test_meta['_m'] = test_meta['model'].map(MODEL_ORDER.index)
    test_meta['_k'] = test_meta['kind'].map(lambda k: KIND_ORDER.index(k) if k in KIND_ORDER else 99)
    test_meta = test_meta.sort_values(['_k', '_m', 'n_train']).drop(columns=['_m', '_k']).reset_index(drop=True)
    pred = pd.concat(frames, ignore_index=True)
    pred = pred[pred['exp_id'].isin(test_meta['exp_id']) & pred['id'].isin(EXPECTED)]
    pred = pred.drop_duplicates(['exp_id', 'id'], keep='last')
    pred = pred.merge(test_meta[['exp_id', 'model', 'kind', 'train_id', 'n_train']], on='exp_id')
    display(test_meta)
    incompletos = test_meta[test_meta['cobertura'] < 1]
    if len(incompletos):
        print(f'[aviso] {len(incompletos)} runs de test incompletos: sus métricas usan solo las imágenes ya inferidas.')
else:
    skip('no hay predicciones de test con p1 (zero-shot de la etapa 1 ni stage2b_test.txt)')

In [ ]:
# ── Extracción de atributos y puntaje por imagen (extractor de la etapa 1) ────
_EXT = {}


def ext_of(text):
    e = _EXT.get(text)
    if e is None:
        e = _EXT[text] = extract(text)
    return e


def score_frame(df):
    rows = []
    for i, t in zip(df['id'], df['prediction']):
        e = ext_of(t)
        s = score(e, TRUTH[i])
        rows.append({**{k: float(v) for k, v in s.items()},
                     'shape_pred': e['shape'] or 'omitido', 'x_pred': e['xpos'] or 'omitido',
                     'y_pred': e['ypos'] or 'omitido', 'color_pred': e['color_cat'] or 'omitido',
                     'n_words': e['n_words']})
    return pd.concat([df.reset_index(drop=True), pd.DataFrame(rows)], axis=1)


if HAVE_TEST:
    scored = score_frame(pred)
    scored = scored.join(truth[['held_out', 'shape_true', 'x_true', 'y_true', 'cat_true', 'canonical', 'reference']], on='id')
    print(f'{len(scored):,} predicciones puntuadas · {len(_EXT):,} textos distintos extraídos')


def run_frame(model, kind, n_train=None):
    """Predicciones puntuadas de un (modelo, clase[, n]) o None."""
    if not HAVE_TEST:
        return None
    m = (scored['model'] == model) & (scored['kind'] == kind)
    if n_train is not None:
        m &= scored['n_train'] == n_train
    out = scored[m]
    if out.empty:
        return None
    if out['exp_id'].nunique() > 1:          # p. ej. dos runs de la misma clase: se exige n_train
        out = out[out['exp_id'] == out['exp_id'].iloc[0]]
    return out


def summarize(df, metrics=METRICS):
    out = {m: df[m].mean() for m in metrics}
    out['n'] = len(df)
    return out


def paired(a, b, metric):
    """Comparación pareada B − A sobre las mismas imágenes. b = A acierta y B falla; c = A falla y B acierta."""
    m = a.set_index('id')[[metric]].join(b.set_index('id')[[metric]], how='inner', lsuffix='_a', rsuffix='_b')
    va, vb = m[f'{metric}_a'], m[f'{metric}_b']
    if metric in BINARY:
        b_, c_ = int(((va == 1) & (vb == 0)).sum()), int(((va == 0) & (vb == 1)).sum())
        p = mcnemar_exact(b_, c_)
    else:                                    # métricas promedio por imagen: z sobre la diferencia pareada
        diff = vb - va
        se = diff.std(ddof=1) / math.sqrt(len(diff)) if len(diff) > 1 else float('nan')
        p = math.erfc(abs(diff.mean() / se) / math.sqrt(2)) if se and se > 0 else 1.0
        b_ = c_ = np.nan
    return {'n': len(m), 'A': va.mean(), 'B': vb.mean(), 'Δ (B−A)': vb.mean() - va.mean(), 'b': b_, 'c': c_, 'p': p}

## 2. Curvas de entrenamiento

Pérdida por paso (entropía cruzada por token en los VLM; InfoNCE contra el banco en VL-JEPA, que **no está en la misma escala**), norma del gradiente, img/s y pico de memoria. Una pérdida que no baja, o que llega a NaN, invalida el run aunque la exactitud en val parezca razonable.

In [ ]:
if HAVE_TRAIN and MAIN_TRAIN:
    fig, axes = plt.subplots(2, 2, figsize=(11, 6.5))
    for m in MODEL_ORDER:
        r = MAIN_TRAIN.get(m)
        if r is None:
            continue
        log = pd.DataFrame(read_jsonl(r.dir / 'train_log.jsonl'))
        if log.empty:
            continue
        kw = dict(color=MODEL_COLORS[m], label=MODEL_LABELS[m], lw=1.4)
        axes[0, 0].plot(log['step'], log['loss'].rolling(5, min_periods=1).mean(), **kw)
        axes[0, 1].plot(log['step'], log['grad_norm'].rolling(5, min_periods=1).mean(), **kw)
        axes[1, 0].plot(log['step'], log['samples_per_s'], **kw)
        axes[1, 1].plot(log['step'], log['peak_mem_gb'], **kw)
    for ax, t in zip(axes.flat, ['Pérdida (media móvil 5)', '‖g‖ antes del recorte', 'Imágenes/s (entrenamiento)',
                                 'Pico de memoria (GB)']):
        ax.set_title(t)
        ax.set_xlabel('paso')
    axes[0, 0].set_yscale('log')
    axes[0, 0].legend()
    fig.suptitle('Entrenamiento principal (objetivo canónico, 1 época, lote 64)', fontweight='semibold')
    fig.tight_layout()
    savefig(fig, '01_curvas_entrenamiento')
    plt.show()
else:
    skip('sin runs principales de entrenamiento')

## 3. Validación durante el entrenamiento

Las mismas 500 imágenes de val en los 4 modelos, evaluadas en el paso 0 (control de paridad: debe reproducir el zero-shot con p1), cada 200 pasos y en el último. La estrella marca el paso elegido (máximo de `shape_pos`). Con $n=500$, el error estándar de una exactitud es $\le 0{,}022$: sirve para elegir el checkpoint, no para comparar modelos (eso se hace en test, §4).

In [ ]:
VAL_METRICS = ['shape_pos', 'shape', 'pos', 'color']
if HAVE_TRAIN and MAIN_TRAIN:
    fig, axes = plt.subplots(1, len(VAL_METRICS) + 1, figsize=(16, 3.6))
    best_rows = []
    for m in MODEL_ORDER:
        r = MAIN_TRAIN.get(m)
        if r is None:
            continue
        val = pd.DataFrame(read_jsonl(r.dir / 'val_log.jsonl'))
        if val.empty:
            continue
        for ax, met in zip(axes, VAL_METRICS + ['val_loss']):
            ax.plot(val['step'], val[met], marker=MODEL_MARKERS[m], ms=4, color=MODEL_COLORS[m], label=MODEL_LABELS[m])
            if r.mejor_paso is not None and met != 'val_loss':
                bv = val.loc[val['step'] == r.mejor_paso, met]
                if len(bv):
                    ax.plot([r.mejor_paso], [bv.iloc[0]], marker='*', ms=13, color=MODEL_COLORS[m], mec=INK, mew=0.6)
        b0 = val.iloc[0]
        bb = val.loc[val['step'] == r.mejor_paso].iloc[0] if r.mejor_paso in set(val['step']) else val.iloc[-1]
        best_rows.append({'modelo': MODEL_LABELS[m], 'paso elegido': int(bb['step']), 'pasos': int(val['step'].max()),
                          **{f'{MLAB[k]} (paso 0)': b0[k] for k in ['shape_pos', 'shape', 'pos', 'color']},
                          **{f'{MLAB[k]} (elegido)': bb[k] for k in ['shape_pos', 'shape', 'pos', 'color']}})
    for ax, met in zip(axes, VAL_METRICS + ['val_loss']):
        ax.set_title(MLAB.get(met, 'Pérdida de val'))
        ax.set_xlabel('paso')
        if met != 'val_loss':
            ax.yaxis.set_major_formatter(PCT)
    axes[0].legend()
    fig.tight_layout()
    savefig(fig, '02_curvas_val')
    plt.show()
    val_best = pd.DataFrame(best_rows).set_index('modelo')
    display(val_best.style.format('{:.1%}', subset=[c for c in val_best.columns if '(' in c]))
    save_table(val_best, 'val_paso_elegido')
else:
    skip('sin val_log de runs principales')

## 4. Test: zero-shot p1 frente a ajustado

Comparación **pareada** sobre las mismas 10 000 imágenes: el mismo prompt (p1) antes y después del ajuste. Para cada atributo binario se usa McNemar exacto, con $b$ = aciertos perdidos y $c$ = aciertos ganados; para las métricas promedio por imagen (`shape_pos`, macro), $z$ sobre la diferencia pareada. Holm controla la tasa de error por familia dentro de cada modelo. Los intervalos son de Wilson al 95 %.

En los modelos con objetivo canónico, *nombre exacto* es 0 por construcción: el modelo nunca dice el nombre literal.

In [ ]:
main_rows, pair_rows = [], []
for m in MODEL_ORDER:
    zs, ft = run_frame(m, 'zero-shot'), run_frame(m, 'principal')
    for kind, df in (('zero-shot', zs), ('ajustado', ft)):
        if df is not None:
            main_rows.append({'modelo': MODEL_LABELS[m], 'run': kind, **summarize(df)})
    if zs is not None and ft is not None:
        rows = [{'modelo': MODEL_LABELS[m], 'métrica': MLAB[k], **paired(zs, ft, k)} for k in METRICS if k != 'name']
        adj = holm([r['p'] for r in rows])
        for r, a in zip(rows, adj):
            r['p_holm'] = a
            r['sig.'] = '✓' if a < ALPHA else ''
        pair_rows += rows

if main_rows:
    main_tab = pd.DataFrame(main_rows).set_index(['modelo', 'run'])
    display(main_tab.style.format({k: '{:.1%}' for k in METRICS} | {'n': '{:,.0f}'}))
    save_table(main_tab, 'test_zeroshot_vs_ajustado')
else:
    skip('sin predicciones de test del run principal ni zero-shot p1')
if pair_rows:
    pair_tab = pd.DataFrame(pair_rows).set_index(['modelo', 'métrica'])
    display(pair_tab.style.format({'A': '{:.1%}', 'B': '{:.1%}', 'Δ (B−A)': '{:+.1%}', 'p': '{:.1e}', 'p_holm': '{:.1e}',
                                   'b': '{:,.0f}', 'c': '{:,.0f}', 'n': '{:,.0f}'}, na_rep='–'))
    save_table(pair_tab, 'test_pareadas_zeroshot_vs_ajustado')

In [ ]:
if main_rows:
    fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(15, 3.8), sharey=True)
    ys = np.arange(len(MAIN_METRICS))
    for ax, m in zip(axes, MODEL_ORDER):
        zs, ft = run_frame(m, 'zero-shot'), run_frame(m, 'principal')
        ax.set_title(MODEL_LABELS[m])
        for df, col, lab, dy in ((zs, NEUTRAL, 'zero-shot p1', 0.12), (ft, MODEL_COLORS[m], 'ajustado', -0.12)):
            if df is None:
                continue
            v = np.array([df[k].mean() for k in MAIN_METRICS])
            n = len(df)
            lo_, hi_ = wilson(v * n, n)
            for j, k in enumerate(MAIN_METRICS):
                if k not in BINARY:
                    se = df[k].std(ddof=1) / math.sqrt(n)
                    lo_[j], hi_[j] = v[j] - Z * se, v[j] + Z * se
            ax.errorbar(v, ys + dy, xerr=[v - lo_, hi_ - v], fmt='o', color=col, ecolor=col, capsize=2, label=lab,
                        mec=INK3 if col == NEUTRAL else col)
        if zs is not None and ft is not None:
            for j, k in enumerate(MAIN_METRICS):
                ax.annotate('', xy=(ft[k].mean(), j - 0.12), xytext=(zs[k].mean(), j + 0.12),
                            arrowprops=dict(arrowstyle='->', color=INK3, lw=0.8))
        ax.set_xlim(0, 1.02)
        ax.xaxis.set_major_formatter(PCT)
    axes[0].set_yticks(ys, [MLAB[k] for k in MAIN_METRICS])
    axes[0].invert_yaxis()
    axes[0].legend(loc='lower left')
    fig.suptitle('Test (n = 10 000): mismo prompt p1, antes y después del ajuste', fontweight='semibold')
    fig.tight_layout()
    savefig(fig, '03_test_antes_despues')
    plt.show()

## 5. Qué cambió con el ajuste

- **Confusión de forma** (filas = verdad, columnas = respuesta; normalizada por fila; *omitido* = no nombra una forma del léxico).
- **Exactitud de posición por celda** de la grilla 3×3 (la perspectiva hace que los objetos de abajo se vean más grandes; el EDA mostró que la fila superior es la más difícil).
- **Transiciones** por imagen: ganadas (zero-shot falla → ajustado acierta) y perdidas.

In [ ]:
SHAPES = [SHAPE_NAMES[i] for i in range(7)]
models_ft = [m for m in MODEL_ORDER if run_frame(m, 'principal') is not None]
if models_ft:
    fig, axes = plt.subplots(2, len(models_ft), figsize=(4.2 * len(models_ft), 7.6), squeeze=False)
    for j, m in enumerate(models_ft):
        for i, (kind, title) in enumerate((('zero-shot', 'zero-shot'), ('principal', 'ajustado'))):
            df = run_frame(m, kind)
            ax = axes[i, j]
            if df is None:
                ax.axis('off')
                continue
            ct = pd.crosstab(df['shape_true'], df['shape_pred']).reindex(index=SHAPES, columns=SHAPES + ['omitido'],
                                                                        fill_value=0)
            heat(ax, ct.div(ct.sum(1), axis=0).values, [s[:4] for s in SHAPES] + ['omit.'], SHAPES, fontsize=6.5)
            ax.set_title(f'{MODEL_LABELS[m]} · {title}  (forma {df["shape"].mean():.0%})')
            ax.tick_params(axis='x', rotation=45)
    fig.tight_layout()
    savefig(fig, '04_confusion_forma')
    plt.show()

    YS, XS = [YPOS_NAMES[i] for i in range(3)], [XPOS_NAMES[i] for i in range(3)]
    fig, axes = plt.subplots(2, len(models_ft), figsize=(3.3 * len(models_ft), 6), squeeze=False)
    for j, m in enumerate(models_ft):
        for i, (kind, title) in enumerate((('zero-shot', 'zero-shot'), ('principal', 'ajustado'))):
            df = run_frame(m, kind)
            ax = axes[i, j]
            if df is None:
                ax.axis('off')
                continue
            g = df.pivot_table(index='y_true', columns='x_true', values='pos', aggfunc='mean').reindex(index=YS, columns=XS)
            heat(ax, g.values, XS, YS, fontsize=8)
            ax.set_title(f'{MODEL_LABELS[m]} · {title}', fontsize=9)
    fig.suptitle('Exactitud de posición (x,y) por celda verdadera', fontweight='semibold')
    fig.tight_layout()
    savefig(fig, '05_posicion_por_celda')
    plt.show()

    trans = []
    for m in models_ft:
        zs, ft = run_frame(m, 'zero-shot'), run_frame(m, 'principal')
        if zs is None:
            continue
        for k in ['shape', 'pos', 'color_adj']:
            p = paired(zs, ft, k)
            jj = zs.set_index('id')[[k]].join(ft.set_index('id')[[k]], how='inner', lsuffix='_a', rsuffix='_b')
            trans.append({'modelo': MODEL_LABELS[m], 'atributo': MLAB[k], 'ganadas': p['c'], 'perdidas': p['b'],
                          'neto': p['c'] - p['b'],
                          'ambos aciertan': int(((jj[f'{k}_a'] == 1) & (jj[f'{k}_b'] == 1)).sum())})
    if trans:
        trans = pd.DataFrame(trans).set_index(['modelo', 'atributo'])
        display(trans)
        save_table(trans, 'transiciones')
else:
    skip('sin predicciones de test del run principal')

## 6. Split composicional

El run composicional no vio en train 14 de las 63 combinaciones forma × posición (cada forma y cada posición sí aparecen por separado). En test, el 22 % de las imágenes tiene una combinación retirada. Se compara contra el run principal **en las mismas imágenes**: el principal vio todas las combinaciones, así que da el techo.

- **Brecha composicional** = exactitud en combinaciones vistas − retiradas, dentro del run composicional. El zero-shot sirve de control: no debería tener brecha, porque no se entrenó con nada.
- La prueba clave es **retiradas, composicional frente a principal** (McNemar). Si no hay diferencia, el modelo compone; si la hay, parte de lo aprendido son pares memorizados.

In [ ]:
comp_rows, comp_tests = [], []
for m in MODEL_ORDER:
    runs = {k: run_frame(m, k) for k in ('zero-shot', 'principal', 'composicional')}
    if runs['composicional'] is None:
        continue
    for kind, df in runs.items():
        if df is None:
            continue
        for grp, sub in (('vistas', df[~df['held_out']]), ('retiradas', df[df['held_out']])):
            comp_rows.append({'modelo': MODEL_LABELS[m], 'run': kind, 'combinaciones': grp,
                              **summarize(sub, ['shape_pos', 'shape', 'pos', 'color'])})
    if runs['principal'] is not None:
        held = lambda d: d[d['held_out']]
        for k in ['shape_pos', 'shape', 'pos']:
            comp_tests.append({'modelo': MODEL_LABELS[m], 'métrica': MLAB[k],
                               **paired(held(runs['principal']), held(runs['composicional']), k)})
if comp_rows:
    comp = pd.DataFrame(comp_rows)
    piv = comp.pivot_table(index=['modelo', 'run'], columns='combinaciones', values='shape_pos')
    piv['brecha (vistas − retiradas)'] = piv['vistas'] - piv['retiradas']
    display(piv.style.format('{:.1%}'))
    save_table(comp.set_index(['modelo', 'run', 'combinaciones']), 'composicional')
    if comp_tests:
        ct = pd.DataFrame(comp_tests)
        ct['p_holm'] = holm(ct['p'])
        ct = ct.set_index(['modelo', 'métrica'])
        print('Imágenes con combinaciones retiradas — A = principal (las vio), B = composicional (no las vio):')
        display(ct.style.format({'A': '{:.1%}', 'B': '{:.1%}', 'Δ (B−A)': '{:+.1%}', 'p': '{:.1e}', 'p_holm': '{:.1e}'},
                                na_rep='–'))
        save_table(ct, 'composicional_retiradas_pareadas')

    mods = list(dict.fromkeys(comp['modelo']))
    fig, axes = plt.subplots(1, len(mods), figsize=(3.6 * len(mods), 3.6), sharey=True, squeeze=False)
    for ax, lab in zip(axes[0], mods):
        sub = comp[comp['modelo'] == lab]
        kinds = [k for k in ('zero-shot', 'principal', 'composicional') if k in set(sub['run'])]
        x = np.arange(len(kinds))
        for dx, grp, col in ((-0.18, 'vistas', SERIES[0]), (0.18, 'retiradas', SERIES[1])):
            v = [sub[(sub['run'] == k) & (sub['combinaciones'] == grp)]['shape_pos'].mean() for k in kinds]
            ax.bar(x + dx, v, width=0.36, color=col, label=grp)
        ax.set_xticks(x, kinds, rotation=15)
        ax.set_title(lab)
        ax.yaxis.set_major_formatter(PCT)
    axes[0][0].set_ylabel('Forma+pos (test)')
    axes[0][0].legend()
    fig.tight_layout()
    savefig(fig, '06_composicional')
    plt.show()
else:
    skip('sin runs composicionales evaluados en test (stage2b_compositional.txt + stage2b_test.txt)')

## 7. Sondas lineales: lo que el modelo sabe frente a lo que dice

Regresión logística sobre la representación (§3.7 del documento de diseño), ajustada con 10 000 imágenes de train y medida en 2 000 de **val**. La exactitud de las respuestas, en cambio, es de **test**: son conjuntos distintos, pero de la misma distribución.

- **Sonda ≫ respuesta zero-shot** → la información está y el LM no la verbaliza: error de formato.
- **Sonda ≈ respuesta ajustada** → el ajuste ya extrae lo que la representación tiene; para mejorar más hay que tocar la visión o el conector.
- **Sonda ajustada > sonda congelada** → el ajuste cambió la representación, no solo la salida.

Se toma, por modelo y atributo, el **mejor rasgo** (capa/posición) de la sonda. Es una cota inferior de lo linealmente decodificable.

In [ ]:
probe_rows = []
for p in sorted(RUNS.glob('probe_*/probe_results.jsonl')):
    probe_rows += read_jsonl(p)
if probe_rows:
    probes = pd.DataFrame(probe_rows)
    probes = probes[probes['model'].isin(MODEL_LABELS)]
    probes['estado'] = np.where(probes['adapter_path'].notna(), 'ajustado', 'congelado')
    probes['modelo'] = probes['model'].map(MODEL_LABELS)
    PROBE_ATTRS = ['shape', 'x', 'y', 'pos', 'color']

    fz = probes[probes['estado'] == 'congelado']
    if len(fz):
        mods = [m for m in MODEL_ORDER if m in set(fz['model'])]
        fig, axes = plt.subplots(1, len(mods), figsize=(3.9 * len(mods), 3.9), squeeze=False)
        for ax, m in zip(axes[0], mods):
            g = fz[fz['model'] == m].pivot_table(index='feature', columns='attribute', values='acc').reindex(columns=PROBE_ATTRS)
            heat(ax, g.values, [MLAB[a] if a in MLAB else a for a in PROBE_ATTRS], list(g.index), fontsize=7.5)
            ax.set_title(f'{MODEL_LABELS[m]} · sonda (congelado)')
            ax.tick_params(axis='x', rotation=30)
        fig.tight_layout()
        savefig(fig, '07_sondas_por_rasgo')
        plt.show()

    best = (probes.sort_values('acc', ascending=False).drop_duplicates(['model', 'estado', 'attribute'])
            .pivot_table(index=['modelo', 'attribute'], columns='estado', values='acc'))
    maj = probes.drop_duplicates(['model', 'attribute']).set_index(['modelo', 'attribute'])['majority_acc']
    best['mayoritaria'] = maj.reindex(best.index)
    for kind, col in (('zero-shot', 'respuesta zero-shot'), ('principal', 'respuesta ajustada')):
        vals = {}
        for m in MODEL_ORDER:
            df = run_frame(m, kind)
            if df is not None:
                for a in PROBE_ATTRS:
                    vals[(MODEL_LABELS[m], a)] = df[a].mean()
        best[col] = pd.Series(vals).reindex(best.index)
    order = [c for c in ['mayoritaria', 'respuesta zero-shot', 'congelado', 'respuesta ajustada', 'ajustado'] if c in best]
    best = best[order].rename(columns={'congelado': 'sonda congelado', 'ajustado': 'sonda ajustado'})
    display(best.style.format('{:.1%}', na_rep='–'))
    save_table(best, 'sondas_vs_respuestas')

    mods = list(dict.fromkeys(best.index.get_level_values(0)))
    fig, axes = plt.subplots(1, len(mods), figsize=(4.2 * len(mods), 3.6), sharey=True, squeeze=False)
    cols = list(best.columns)
    palette = {'mayoritaria': NEUTRAL, 'respuesta zero-shot': SERIES[1], 'sonda congelado': SERIES[0],
               'respuesta ajustada': SERIES[2], 'sonda ajustado': SERIES[6]}
    w = 0.8 / len(cols)
    for ax, lab in zip(axes[0], mods):
        sub = best.loc[lab].reindex(PROBE_ATTRS)
        x = np.arange(len(PROBE_ATTRS))
        for i, c in enumerate(cols):
            ax.bar(x - 0.4 + w / 2 + i * w, sub[c].values, width=w, color=palette.get(c, INK3), label=c)
        ax.set_xticks(x, [MLAB[a] for a in PROBE_ATTRS], rotation=20)
        ax.set_title(lab)
        ax.yaxis.set_major_formatter(PCT)
    axes[0][0].legend(fontsize=7.5)
    fig.tight_layout()
    savefig(fig, '08_sondas_vs_respuestas')
    plt.show()
else:
    skip('sin resultados de sondas (stage2b_probe.txt / stage2b_probe_ft.txt)')

## 8. Controles: objetivo original, bf16 y semilla

Cada control se compara pareado contra el run principal del mismo modelo, en test:

- **original − canónico:** costo de aprender las 5 plantillas y los ~100 nombres de color, con todo lo demás igual. Con el objetivo original, *nombre exacto* deja de ser 0 por construcción.
- **bf16 − NF4:** costo de cuantizar la base durante el ajuste.
- **semilla 1 − semilla 0:** ruido de entrenamiento. Una diferencia entre modelos menor que esta no es interpretable.

In [ ]:
ctrl_rows = []
for kind in ('original', 'bf16', 'semilla'):
    for m in MODEL_ORDER:
        a, b = run_frame(m, 'principal'), run_frame(m, kind)
        if a is None or b is None:
            continue
        for k in ['shape_pos', 'shape', 'pos', 'color', 'color_adj', 'name', 'macro']:
            ctrl_rows.append({'control': kind, 'modelo': MODEL_LABELS[m], 'métrica': MLAB[k], **paired(a, b, k)})
if ctrl_rows:
    ctrl = pd.DataFrame(ctrl_rows)
    ctrl['p_holm'] = ctrl.groupby(['control', 'modelo'])['p'].transform(lambda s: holm(s.values))
    ctrl = ctrl.set_index(['control', 'modelo', 'métrica'])
    print('A = principal (canónico, NF4, semilla 0) · B = control')
    display(ctrl.style.format({'A': '{:.1%}', 'B': '{:.1%}', 'Δ (B−A)': '{:+.1%}', 'p': '{:.1e}', 'p_holm': '{:.1e}',
                               'b': '{:,.0f}', 'c': '{:,.0f}', 'n': '{:,.0f}'}, na_rep='–'))
    save_table(ctrl, 'controles')
else:
    skip('sin controles evaluados en test (stage2b_ablations.txt + stage2b_test.txt)')

zs_bf = {m: run_frame(m, 'zero-shot bf16') for m in MODEL_ORDER}
if any(v is not None for v in zs_bf.values()):
    print('Zero-shot bf16 disponible para:', [MODEL_LABELS[m] for m, v in zs_bf.items() if v is not None])
else:
    print('[nota] no hay zero-shot bf16 con p1: la ablación bf16 solo se compara contra el ajustado NF4 (ver docs §5).')

## 9. Costo y formato de las respuestas

**Costo.** Mismo número de imágenes y de pasos para todos, pero no el mismo costo: LLaVA-OV procesa ~20× más tokens visuales que Qwen3.5. Se reportan horas de GPU, img/s, pico de memoria y parámetros entrenables.

**Formato.** Indica si el modelo ajustado adoptó la plantilla canónica: la fracción de respuestas que **calzan con la plantilla**, la fracción **idéntica a su objetivo** (exactitud de oración completa), la diversidad y el largo. Una diversidad muy baja con una exactitud alta sugiere colapso a pocas respuestas: hay que revisarla junto con §5.

In [ ]:
if HAVE_TRAIN:
    cost = []
    for r in train_runs.itertuples():
        log = pd.DataFrame(read_jsonl(r.dir / 'train_log.jsonl'))
        cost.append({'modelo': r.label, 'clase': r.kind, 'n_train': r.n_train, 'épocas': r.epochs,
                     'pasos': r.paso, 'h entrenamiento': r.train_h, 'h evaluación': r.eval_h,
                     'img/s (mediana)': log['samples_per_s'].median() if len(log) else np.nan,
                     'pico GB': log['peak_mem_gb'].max() if len(log) else np.nan,
                     'LoRA (M)': (r.lora or 0) / 1e6, 'conector (M)': (r.conector or 0) / 1e6, 'estado': r.estado})
    cost = pd.DataFrame(cost).set_index(['modelo', 'clase', 'n_train'])
    display(cost.style.format({'h entrenamiento': '{:.2f}', 'h evaluación': '{:.2f}', 'img/s (mediana)': '{:.1f}',
                               'pico GB': '{:.1f}', 'LoRA (M)': '{:.2f}', 'conector (M)': '{:.2f}'}, na_rep='–'))
    save_table(cost, 'costo')

CANON_RE = re.compile(r'^A (red|orange|yellow|green|cyan|blue|purple|pink) (teapot|hare|dragon|cow|armadillo|horse|head) '
                      r'is at the (top|mid|bottom)-(left|center|right) of the image\.$')
if HAVE_TEST:
    fmt_rows = []
    for (exp, m, kind), g in scored.groupby(['exp_id', 'model', 'kind'], sort=False):
        target = g['reference'] if TRAIN_KIND.get(g['train_id'].iloc[0]) == 'original' else g['canonical']
        fmt_rows.append({'modelo': MODEL_LABELS[m], 'run': kind, 'n_train': g['n_train'].iloc[0],
                         'plantilla canónica': g['prediction'].str.strip().str.fullmatch(CANON_RE.pattern).mean(),
                         'idéntica al objetivo': (g['prediction'].str.strip() == target).mean(),
                         'respuestas distintas': g['prediction'].nunique() / len(g),
                         'palabras (mediana)': g['n_words'].median()})
    fmt = pd.DataFrame(fmt_rows).set_index(['modelo', 'run', 'n_train'])
    display(fmt.style.format({'plantilla canónica': '{:.1%}', 'idéntica al objetivo': '{:.1%}',
                              'respuestas distintas': '{:.1%}', 'palabras (mediana)': '{:.0f}'}, na_rep='–'))
    save_table(fmt, 'formato_respuestas')

## 10. Ejemplos cualitativos y síntesis

Imágenes en que el zero-shot falla en forma o posición y el modelo ajustado acierta en ambas, con la respuesta de cada uno.

In [ ]:
ex_models = [m for m in MODEL_ORDER if run_frame(m, 'principal') is not None and run_frame(m, 'zero-shot') is not None]
if ex_models:
    root = data_root(VARIANT)
    fig, axes = plt.subplots(len(ex_models), N_EXAMPLES, figsize=(4.6 * N_EXAMPLES, 4.4 * len(ex_models)), squeeze=False)
    for i, m in enumerate(ex_models):
        zs = run_frame(m, 'zero-shot').set_index('id')
        ft = run_frame(m, 'principal').set_index('id')
        both = zs.join(ft, how='inner', lsuffix='_zs', rsuffix='_ft')
        cand = both[(both['shape_pos_zs'] < 1) & (both['shape_pos_ft'] == 1)].index.tolist()
        pick = list(RNG.choice(cand, size=min(N_EXAMPLES, len(cand)), replace=False)) if cand else []
        for j in range(N_EXAMPLES):
            ax = axes[i, j]
            ax.axis('off')
            if j >= len(pick):
                continue
            iid = pick[j]
            try:
                from PIL import Image
                ax.imshow(Image.open(root / truth.loc[iid, 'image_path']).convert('RGB'))
            except Exception as e:
                ax.text(0.5, 0.5, f'imagen no disponible\n{e}', ha='center', va='center', fontsize=7)
            t = truth.loc[iid]
            wrap = lambda s, n=60: '\n'.join(textwrap.wrap(str(s), n)[:3])
            ax.set_title(f'{MODEL_LABELS[m]} · verdad: {t.cat_true} {t.shape_true}, {t.y_true}-{t.x_true}\n'
                         f'ZS: {wrap(both.loc[iid, "prediction_zs"])}\nFT: {wrap(both.loc[iid, "prediction_ft"])}',
                         fontsize=7, loc='left')
    fig.tight_layout()
    savefig(fig, '09_ejemplos')
    plt.show()
else:
    skip('sin pares zero-shot / ajustado en test para ejemplos')

In [ ]:
# ── Tabla de síntesis (una fila por modelo) ───────────────────────────────────
synth = []
for m in MODEL_ORDER:
    row = {'modelo': MODEL_LABELS[m]}
    for kind, tag in (('zero-shot', 'ZS'), ('principal', 'FT')):
        df = run_frame(m, kind)
        for k in ['shape_pos', 'shape', 'pos', 'color', 'macro']:
            row[f'{MLAB[k]} {tag}'] = df[k].mean() if df is not None else np.nan
    comp_df = run_frame(m, 'composicional')
    if comp_df is not None:
        row['brecha composicional'] = (comp_df[~comp_df['held_out']]['shape_pos'].mean()
                                       - comp_df[comp_df['held_out']]['shape_pos'].mean())
    if probe_rows:
        fz = probes[(probes['model'] == m) & (probes['estado'] == 'congelado')]
        for a in ['shape', 'pos']:
            row[f'sonda {MLAB[a]}'] = fz[fz['attribute'] == a]['acc'].max() if len(fz) else np.nan
    if HAVE_TRAIN and m in MAIN_TRAIN:
        row['h GPU (train)'] = MAIN_TRAIN[m].train_h
    synth.append(row)
synth = pd.DataFrame(synth).set_index('modelo').dropna(axis=1, how='all')
pct_cols = [c for c in synth.columns if not c.startswith('h GPU')]
display(synth.style.format({c: '{:.1%}' for c in pct_cols} | {'h GPU (train)': '{:.2f}'}, na_rep='–'))
save_table(synth, 'sintesis')
print(f'Tablas en {TAB_DIR.relative_to(ROOT)} · figuras en {FIG_DIR.relative_to(ROOT)}')